## Currency Conversion Tool

In [52]:
from langchain.tools import tool
import requests
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import json

In [53]:
from dotenv import load_dotenv
load_dotenv()
import os

In [54]:
api_key=os.getenv('V6_EXCHANGE_API_KEY')

In [75]:
## creating tools:
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches the currency conversion factor between a base currency and a target currency."""
    url=f'https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency}/{target_currency}'
    response=requests.get(url)
    return response.json()



In [76]:
@tool
def convert(base_currency_value: int, conversion_factor: Annotated[float, InjectedToolArg])-> float:
    """given a currency conversion rate this function calculates the target currency value from a given base currency value."""
    return base_currency_value * conversion_factor

In [94]:
@tool
def convert_currency(base_currency_value: int, conversion_factor: Annotated[float, InjectedToolArg])-> float:
    """given a currency conversion rate this function calculates the target currency value from a given base currency value."""
    return base_currency_value * conversion_factor

In [95]:
get_conversion_factor.invoke({'base_currency':"USD",'target_currency': "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1748390402,
 'time_last_update_utc': 'Wed, 28 May 2025 00:00:02 +0000',
 'time_next_update_unix': 1748476802,
 'time_next_update_utc': 'Thu, 29 May 2025 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 85.3663}

In [96]:
convert.invoke({'base_currency_value':10,'conversion_factor':85.3412})

853.412

In [97]:
## tool binding
from langchain_openai import AzureChatOpenAI
model=AzureChatOpenAI(
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    azure_deployment=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION'),
    model=os.getenv('AZURE_OPENAI_MODEL_NAME')
)

In [98]:
model_with_tools= model.bind_tools([get_conversion_factor,convert])

In [99]:
messages= [HumanMessage(content="What is the conversion factor between  USD to INR, and based on that can you help me to convert 10 usd to inr value using convert function from tool calling?")]

In [100]:
messages

[HumanMessage(content='What is the conversion factor between  USD to INR, and based on that can you help me to convert 10 usd to inr value using convert function from tool calling?', additional_kwargs={}, response_metadata={})]

In [101]:
ai_message=model_with_tools.invoke(messages)
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_jDSdOP9nbkRuB07XjqC8ZZug', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 130, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_07e970ab25', 'id': 'chatcmpl-Bc5xqLwkEvebaqNeU5YeXa6rD3DM5', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filte

In [102]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_jDSdOP9nbkRuB07XjqC8ZZug',
  'type': 'tool_call'}]

In [51]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated